Интернет-магазин «В один клик» продаёт разные товары: для детей, для дома, мелкую бытовую технику, косметику и даже продукты. Отчёт магазина за прошлый период показал, что активность покупателей начала снижаться. Привлекать новых клиентов уже не так эффективно: о магазине и так знает большая часть целевой аудитории. 

Необходимо разработать решение, которое позволит персонализировать предложения постоянным клиентам, чтобы увеличить их покупательскую активность.

Предоставдены данные в табличной форме. 

+ `market_file.csv` - содержит данные о поведении покупателя на сайте, о коммуникациях с покупателем и его продуктовом поведении.

    + `id` — номер покупателя в корпоративной базе данных.
    + `Покупательская активность` — рассчитанный класс покупательской активности (целевой признак): «снизилась» или «прежний уровень».
    + `Тип сервиса` — уровень сервиса, например «премиум» и «стандарт».
    + `Разрешить сообщать` — информация о том, можно ли присылать покупателю дополнительные предложения о товаре. Согласие на это даёт покупатель.
    + `Маркет_актив_6_мес` — среднемесячное значение маркетинговых коммуникаций компании, которое приходилось на покупателя за последние 6 месяцев. Это значение показывает, какое число рассылок, звонков, показов рекламы и прочего приходилось на клиента.
    + `Маркет_актив_тек_мес` — количество маркетинговых коммуникаций в текущем месяце.
    + `Длительность` — значение, которое показывает, сколько дней прошло с момента регистрации покупателя на сайте.
    + `Акционные_покупки` — среднемесячная доля покупок по акции от общего числа покупок за последние 6 месяцев.
    + `Популярная_категория` — самая популярная категория товаров у покупателя за последние 6 месяцев.
    + `Средний_просмотр_категорий_за_визит` — показывает, сколько в среднем категорий покупатель просмотрел за визит в течение последнего месяца. Неоплаченные_продукты_штук_квартал — общее число неоплаченных товаров в корзине за последние 3 месяца.
    + `Ошибка_сервиса` — число сбоев, которые коснулись покупателя во время посещения сайта.
    + `Страниц_за_визит` — среднее количество страниц, которые просмотрел покупатель за один визит на сайт за последние 3 месяца.

+ `market_money.csv` - с данными о выручке, которую получает магазин с покупателя, то есть сколько покупатель всего потратил за период взаимодействия с сайтом.

    + `id` — номер покупателя в корпоративной базе данных.
    + `Период` — название периода, во время которого зафиксирована выручка. Например, 'текущий_месяц' или 'предыдущий_месяц'.
    + `Выручка` — сумма выручки за период.

+ `market_time.csv` - с данными о времени (в минутах), которое покупатель провёл на сайте в течение периода.

    + `id` — номер покупателя в корпоративной базе данных.
    + `Период` — название периода, во время которого зафиксировано общее время.
    + `минут` — значение времени, проведённого на сайте, в минутах.

+ `money.csv` - с данными о среднемесячной прибыли покупателя за последние 3 месяца: какую прибыль получает магазин от продаж каждому покупателю.

    + `id` — номер покупателя в корпоративной базе данных.
    + `Прибыль` — значение прибыли.


Разобъем задачу на два этапа:

1. Разработаем модель, которая предскажет вероятность снижения покупательской активности.
2. Выделим сегмент покупателей, проанализируем его и предложим, как увеличить его покупательскую активность, используя данные моделирования, данные о прибыли покупателей и исходные данные (если понадобятся). Возможные сегменты:
+ Группа клиентов с максимальной долей покупок по акции и высокой вероятностью снижения покупательской активности.
+ Группа клиентов, которые покупают только технику, то есть товары с длинным жизненным циклом.
+ Группа клиентов, которые покупают товары из категории «Товары для себя» (новая категория, которую можно выделить на основе текущих) или «Товары для детей».
+ Группа клиентов с высокой вероятностью снижения покупательской активности и наиболее высокой прибыльностью.

План работы:

1. Загрузка данных
2. Предобработка данных
3. Исследовательский анализ данных
4. Объединение таблиц
5. Корреляционный анализ
6. Использование пайплайнов
7. Анализ важности признаков
8. Сегментация покупателей
9. Общий вывод


## Загрузка данных

 **Установка пакетов**
 

In [ ]:
# Обновление библиотек для устранения ошибок
#---
import numba
import sys
if (numba.__version__) < ('0.60.0'):
    !{sys.executable} -m pip install numba --upgrade -q
    print("Numba Обновлена до версии : {}".format(numba.__version__))
#---
import sklearn
if (sklearn.__version__) < ('1.6.1'):
    !{sys.executable} -m pip install --upgrade scikit-learn -q
    print("Scikit-learn Обновлена до версии : {}".format(sklearn.__version__))
#---

In [ ]:
# Проверка окружающей среды
import os
try:
    print('Локальная окружающая среда =>', os.environ['CONDA_DEFAULT_ENV'])
except:
    print('Код выполняется на сервере')

 **Импорт библиотек**

In [ ]:
# Основные библиотеки
import numpy as np
import pandas  as pd
import requests
import warnings

# Графики
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as st
from scipy.stats import shapiro
from scipy.stats._continuous_distns import _distn_names

#---
try:
    import phik
    from phik import phik_matrix
except:
    !{sys.executable} -m pip install phik -q
    import phik
    from phik import phik_matrix
#---
try:
    import shap
except:
    !{sys.executable} -m pip install shap -q
    import shap
#---

# Модели
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (train_test_split,
                                    GridSearchCV,
                                    RandomizedSearchCV
)
from sklearn.metrics import (accuracy_score,
                            confusion_matrix,
                            precision_score,
                            recall_score,
                            roc_auc_score
)
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler,
                                    LabelEncoder,
                                    OneHotEncoder,
                                    OrdinalEncoder,
                                    RobustScaler,
                                    StandardScaler
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC


In [ ]:
# Постоянные и настройки системы
ALFA = 0.05
RANDOM_STATE = 42

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

warnings.filterwarnings("ignore")

**Загрузка данных**

### загрузка для market_file

In [ ]:
pth11 = '/datasets/market_file.csv'
pth12 = 'https://code.s3.yandex.net/datasets/market_file.csv'

if os.path.exists(pth11):
    market_file = pd.read_csv(pth11)
elif requests.get(pth12).status_code == 200:
    market_file = pd.read_csv(pth12)
else:
    print('Что-то не так')
    
market_file.head()

### загрузка для market_money

In [ ]:
pth11 = '/datasets/market_money.csv'
pth12 = 'https://code.s3.yandex.net/datasets/market_money.csv'

if os.path.exists(pth11):
    market_money = pd.read_csv(pth11)
elif requests.get(pth12).status_code == 200:
    market_money = pd.read_csv(pth12)
else:
    print('Что-то не так')
    
market_money.head()

### Загрузка для market_time

In [ ]:
pth11 = '/datasets/market_time.csv'
pth12 = 'https://code.s3.yandex.net/datasets/market_time.csv'

if os.path.exists(pth11):
    market_time = pd.read_csv(pth11)
elif requests.get(pth12).status_code == 200:
    market_time = pd.read_csv(pth12)
else:
    print('Что-то не так')
    
market_time.head()

### Загрузка для money

In [ ]:
pth11 = '/datasets/money.csv'
pth12 = 'https://code.s3.yandex.net/datasets/money.csv'

if os.path.exists(pth11):
    money = pd.read_csv(pth11, sep = ';', decimal = ',')
elif requests.get(pth12).status_code == 200:
    money = pd.read_csv(pth12, sep = ';', decimal = ',')
else:
    print('Что-то не так')
    
money.head()

## Предобработка данных

### Предобработка market_file

In [ ]:
# market_file
market_file.head()

In [ ]:
# Изменение названий столбцов
market_file.columns = map(str.lower, market_file.columns)
market_file.columns = market_file.columns.str.replace(' ', '_')
market_file.columns

In [ ]:
display(market_file["покупательская_активность"].unique())
display(market_file["разрешить_сообщать"].unique())
display(market_file["популярная_категория"].unique())
market_file["тип_сервиса"].unique()

In [ ]:
# Исправление данных
market_file["тип_сервиса"] = market_file["тип_сервиса"].replace(['стандартт'],'стандарт')
market_file["популярная_категория"] = market_file["популярная_категория"].replace(['Косметика и аксесуары'],'Косметика и аксессуары')
display(market_file["популярная_категория"].unique())
market_file["тип_сервиса"].unique()

In [ ]:
#проверка на дубликаты
print('Явных дубликатов:', market_file.duplicated().sum(), '\n')
market_file.info()

### Предобработка market_money

In [ ]:
display(market_money["Период"].unique())
market_money.info()

In [ ]:
# Изменение названий столбцов
market_money.columns = map(str.lower, market_money.columns)
print('Явных дубликатов:', market_money.duplicated().sum(), '\n')
market_money.info()

### Предобработка market_time

In [ ]:
# Изменение названий столбцов
market_time.columns = map(str.lower, market_time.columns)
display(market_time["период"].unique())
market_time.info()

In [ ]:
# Исправление данных
market_time["период"] = market_time["период"].replace(['предыдцщий_месяц'],'предыдущий_месяц')
print('Явных дубликатов:', market_time.duplicated().sum(), '\n')
market_time.info()

### Предобработка money

In [ ]:
money.columns = map(str.lower, money.columns)
print('Явных дубликатов:', money.duplicated().sum(), '\n')
money.info()

***Вывод по предобработке данных***

В представленных данных пропуски не обнаружены

Произведено изменение названий столбцов:
+ перевод в строчный формат
+ замена пробелов на подчеркивание

В `market_file` 
+ в колонке `тип_сервиса` данные `стандартт` заменены на `стандарт`
+ в колонке `популярная_категория` данные `Косметика и аксесуары` заменены на `Косметика и аксессуары`

В `market_money`
+ в колонке `период`данные `препредыдущий_месяц` заменены на `предыдущий_месяц`

В `market_time`
+ в колонке `период` данные `предыдцщий_месяц` заменены на `предыдущий_месяц`

##  Исследовательский анализ данных

In [ ]:
market_file.describe()

**Анализ market_file[´маркет_актив_6_мес´]**

In [ ]:
# Раздельные графики
plt.subplots(figsize = (10,5))
sns.histplot(data=market_file, x="маркет_актив_6_мес", bins=50)
plt.title('Гистограмма маркетинговых коммуникаций в месяц')
plt.xlabel('Месяц')
plt.ylabel('Количество')
plt.show()

plt.subplots(figsize = (10,5))
sns.boxplot(x=market_file["маркет_актив_6_мес"])
plt.title('Boxplot маркетинговых коммуникаций в месяц')
plt.xlabel('Месяц')
plt.ylabel('маркет_актив_6_мес')
plt.show()

In [ ]:
# Совместный график
def f_hist_box(data, bins, xlabel, ylabel):
    f, (ax_box, ax_hist) = plt.subplots(2, sharex=True, figsize=(14,5), gridspec_kw={"height_ratios": (.15, .85)})
    sns.boxplot(data=[data], orient="h", ax=ax_box)
    ax_box.set_title(f'Боксплот столбца {data.name}')
    
    sns.histplot(data=data, color='blue', bins=bins, common_norm=False, alpha=0.5, kde=True, label=data.name, ax=ax_hist)
    ax_hist.legend(loc='upper right')
    ax_hist.set_title(f'Гистограмма столбца {data.name} и  плотность ядра KDE')
    ax_hist.set_xlabel(xlabel)
    ax_hist.set_ylabel(ylabel) 
    
    plt.show()
    
f_hist_box(
    data=market_file['маркет_актив_6_мес'],
    bins=50, 
    xlabel='Месяц',
    ylabel='Количество'
)

In [ ]:
# Совместный график с подбором модели и проверкой данных на нормальность
# Функция перебора моделей
def f_best_fit_distribution(data, bins=200, ax=None):
    # Моделируем данные, находя наиболее подходящее распределение для данных
    # Получаем histogram для оригинальных данных
    y, x = np.histogram(data, bins=bins, density=True)
    x = (x + np.roll(x, -1))[:-1] / 2.0

    # Лучшая модель
    best_distributions = []

    # Оцениваем параметры распределения данных
    for ii, distribution in enumerate([d for d in _distn_names if not d in ['levy_stable', 'studentized_range']]):
        
        sys.stdout.write('\r' + 'Обрабатываем {:>3} распределение из {:<3} : {}'.format( ii+1, len(_distn_names), distribution ))
        
        distribution = getattr(st, distribution)

        # Постарайтесь соответствовать распределению
        try:
            # Игнорируем предупреждения от неподходящих данных
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore')
                
                # Соответствие данным
                params = distribution.fit(data)

                # Отдельные части параметров
                arg = params[:-2]
                loc = params[-2]
                scale = params[-1]
                
                # Рассчитаем подогнанный PDF-файл и ошибку с помощью подогнанного распределения
                pdf = distribution.pdf(x, loc=loc, scale=scale, *arg)
                sse = np.sum(np.power(y - pdf, 2.0))
                
                # если ось проходит в добавлении к графику
                try:
                    if ax:
                        pd.Series(pdf, x).plot(ax=ax)
                    end
                except Exception:
                    pass

                # определяем, лучше ли это распределение
                best_distributions.append((distribution, params, sse))
        
        except Exception:
            pass

    
    return sorted(best_distributions, key=lambda x:x[2])

# Функция распределения вероятностей распределений
def f_make_pdf(dist, params, size=10000):
    
    # Отдельные части параметров
    arg = params[:-2]
    loc = params[-2]
    scale = params[-1]

    # Получаем начальные и конечные точки распространения
    start = dist.ppf(0.01, *arg, loc=loc, scale=scale) if arg else dist.ppf(0.01, loc=loc, scale=scale)
    end = dist.ppf(0.99, *arg, loc=loc, scale=scale) if arg else dist.ppf(0.99, loc=loc, scale=scale)

    # Создаем PDF-файл и превращаем его в серию pandas
    x = np.linspace(start, end, size)
    y = dist.pdf(x, loc=loc, scale=scale, *arg)
    pdf = pd.Series(y, x)

    return pdf

# Функция построения боксплот, гистограммы и сравнительного подбора
def f_box_hist_relative(data, xlabel, ylabel, bins=50):
    # Находим наиболее подходящее распределение
    best_dist = f_best_fit_distribution(data, 200)[0]
    # Создаем PDF-файл с лучшими параметрами
    pdf = f_make_pdf(best_dist[0], best_dist[1])
    param_names = (best_dist[0].shapes + ', loc, scale').split(', ') if best_dist[0].shapes else ['loc', 'scale']
    param_str = ', '.join(['{}={:0.2f}'.format(k,v) for k,v in zip(param_names, best_dist[1])])
    dist_str = '{}({})'.format(best_dist[0].name, param_str)

    f, (ax_box, ax_hist, ax_relative) = plt.subplots(3, sharex=True, figsize=(10,10), gridspec_kw={"height_ratios": (.10, .45, .45)})
     
    sns.boxplot(data=data, orient="h", ax=ax_box)
    ax_box.set_title(f'Боксплот столбца {data.name}')

    sns.histplot(data=data, color='blue', bins=bins, common_norm=False, alpha=0.5, kde=True, label=data.name, ax=ax_hist)
    ax_hist.legend(loc='upper right')
    ax_hist.set_title(f'Гистограмма столбца {data.name} и  плотность ядра KDE')
    ax_hist.set_ylabel(ylabel) 

    sns.histplot(data=data, color='blue', bins=bins, stat='density', common_norm=False, alpha=0.5, kde=True, label=data.name, ax=ax_relative)
    sns.lineplot(data=pdf, color='red', label=best_dist[0].name, ax=ax_relative)
    
    ax_relative.set_xlim(data.min(), data.max())
    ax_relative.legend(loc='upper right')
    ax_relative.set_title('Подобранная модель из библиотеки scipy.stats и её параметры \n' + dist_str)
    ax_relative.set_xlabel(xlabel)
    ax_relative.set_ylabel(ylabel + ' относительное')
    plt.show();

    # Тест на нормальность (Шапиро-Уилка)
    shapiro_test = shapiro(data)
    print(f"Значения теста Шапиро-Уилка: статистика= {shapiro_test.statistic:.4f}, P-значение= {shapiro_test.pvalue:.4f}")
    # Если p-значение < ALFA, распределение не является нормальным
    if shapiro_test.pvalue < ALFA:
        print(f'Данные при пороге = {ALFA} не выглядят нормально (отвергаем нулевую гипотезу)')
    else:
        print(f'Данные при пороге = {ALFA} выглядят нормально (не отвергаем нулевую гипотезу)')

    return print(f'\nНаиболее подходящая модель: "{dist_str}" из библиотеки scipy.stats\
    \nФункцию плотности и вероятности можно посмотреть в интернете для описания')

# Функция пределов для очистки от выбросов
def f_clean(data):
    Q1 = data.quantile(0.25)  # Рассчитываем 25-й процентиль  
    Q3 = data.quantile(0.75)  # Определяем 75-й процентиль  
    IQR = Q3 - Q1  # Рассчитываем IQR
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

In [ ]:
f_box_hist_relative(data=market_file['маркет_актив_6_мес'],
                    xlabel='Месяц',
                    ylabel='Количество',
                    bins=50,
                    )

**Анализ market_file[´маркет_актив_6_мес´] - дискретный признак**

In [ ]:
plt.subplots(figsize = (10,5))
sns.countplot(data =  market_file, x = 'маркет_актив_тек_мес')
plt.title('Маркетинговые коммуникаций в текущем месяце')
plt.xlabel('Количество коммуникаций')
plt.ylabel('Количество покупателей')
plt.show();

**Анализ market_file['длительность']**

In [ ]:
# Раздельные графики
plt.subplots(figsize = (10,5))
sns.histplot(data=market_file, x='длительность', bins=100)
plt.title('Гистограмма прошедших дней после регистрации покупателя')
plt.xlabel('Количество дней')
plt.ylabel('Количество покупателей')
plt.show()

plt.subplots(figsize = (10,5))
sns.boxplot(x=market_file['длительность'])
plt.title('Boxplot прошедших дней после регистрации покупателя')
plt.xlabel('Количество дней')
plt.ylabel('длительность')
plt.show()

In [ ]:
# Совместный график
f_hist_box(
    data=market_file['длительность'],
    bins=100, 
    xlabel='Количество дней',
    ylabel='Количество покупателей'
)

In [ ]:
# Совместный график с подбором модели и проверкой данных на нормальность
f_box_hist_relative(data=market_file['длительность'],
                    xlabel='Количество дней',
                    ylabel='Количество покупателей',
                    bins=100,
                    )

**Анализ market_file['акционные_покупки']**

In [ ]:
# Раздельные графики
plt.subplots(figsize = (10,5))
sns.histplot(data=market_file, x='акционные_покупки', bins=100)
plt.title('Гистограмма среднемесячной доли покупок по акциям')
plt.xlabel('Доля акционных покупок')
plt.ylabel('Количество покупателей')
plt.show()

plt.subplots(figsize = (10,5))
sns.boxplot(x=market_file['акционные_покупки'])
plt.title('Boxplot среднемесячной доли покупок по акциям')
plt.xlabel('Доля акционных покупок')
plt.ylabel('акционные_покупки')
plt.show()

In [ ]:
# Совместный график
f_hist_box(data=market_file['акционные_покупки'],
           bins=100,
           xlabel='Доля акционных покупок',
           ylabel='Количество покупателей'
)

In [ ]:
f_box_hist_relative(data=market_file['акционные_покупки'],
                    xlabel='Доля акционных покупок',
                    ylabel='Количество покупателей',
                    bins=100,
                    )

**Анализ market_file['акционные_покупки'] - дискретный признак**

In [ ]:
plt.subplots(figsize = (10,5))
sns.countplot(x = 'средний_просмотр_категорий_за_визит', data =  market_file)
plt.title('Средний просмотр категорий за визит')
plt.xlabel('Количество просмотренных категорий')
plt.ylabel('Количество просмотренных категорий')
plt.show()

**Анализ market_file['неоплаченные_продукты_штук_квартал'] - дискретный признак**

In [ ]:
plt.subplots(figsize = (10,5))
sns.countplot(x = 'неоплаченные_продукты_штук_квартал', data =  market_file)
plt.title('Неоплаченные товары в корзине за последние 3 месяца')
plt.xlabel('Неоплаченных товаров')
plt.ylabel('Количество покупателей')
plt.show()

**Анализ market_file['ошибка_сервиса'] - дискретный признак**

In [ ]:
plt.subplots(figsize = (10,5))
sns.countplot(x = 'ошибка_сервиса', data =  market_file)
plt.title('Количество сбоев при посещении сайта')
plt.xlabel('Количество сбоев')
plt.ylabel('Количество покупателей')
plt.show()

**Анализ market_file['страниц_за_визит'] - дискретный признак**

In [ ]:
plt.subplots(figsize = (10,5))
sns.countplot(x = 'страниц_за_визит', data =  market_file)
plt.title('Количество просмотренных страниц за визит за последние 3 месяца')
plt.xlabel('Количество просмотренных страниц')
plt.ylabel('Количество покупателей')
plt.show()

Анализ строковых значений `market_file`

In [ ]:
market_file.describe(include='object')

**Анализ market_file['покупательская_активность']**

In [ ]:
# Диаграмма
(
market_file
    .pivot_table(index='покупательская_активность',values='id',aggfunc='count')
    .plot.pie(y='id', autopct='%1.0f%%', figsize=(10,10), 
              label='Процентное отношение покупательской активности')
)
plt.title('Диаграмма покупательской активности')
plt.show()

In [ ]:
# Процентное отношение и количество
def f_countplot(f, n, suptitle, xlabel, ylabel):
    fig, (ax1) = plt.subplots(1, 1, figsize=(10, 5))
    fig.tight_layout(h_pad=3)
    fig.suptitle(suptitle, x=0.5, y=1.01, fontsize=16)
    
    sns.countplot(y=n, data=f, palette='colorblind')
    
    ax1.set_xlabel(ylabel)
    ax1.set_ylabel(xlabel)

    total = len(f[n])
    for p in ax1.patches:
        percentage = '{:.1f}%'.format(100 * p.get_width() / total)
        x = p.get_x() + p.get_width() + 0.02
        y = p.get_y() + p.get_height() / 2
        ax1.annotate(percentage, (x, y))
    plt.show()
    
f_countplot(
    f = market_file,
    n = 'покупательская_активность',
    suptitle = 'Количество и процент покупательской активности',
    xlabel='Категории',
    ylabel='Количество и процент'
)

В данных присутствует дисбаланс целевой переменной  62:38, это нужно учитывать при выборе метрики для отбора лучшей модели

**Анализ market_file['тип_сервиса']**

In [ ]:
# Диаграмма
(
market_file
    .pivot_table(index='тип_сервиса',values='id',aggfunc='count')
    .plot.pie(y='id', autopct='%1.0f%%', figsize=(10,10), 
              label='Процентное отношение типа сервиса')
)
plt.title('Диаграмма типа сервиса')
plt.show()

In [ ]:
# Процентное отношение и количество
f_countplot(
    f = market_file,
    n = 'тип_сервиса',
    suptitle = 'Количество и процент тип_сервиса',
    xlabel='Категории',
    ylabel='Количество и процент'
)

**Анализ market_file['разрешить_сообщать']**

In [ ]:
# Диаграмма
(
market_file
    .pivot_table(index='разрешить_сообщать',values='id',aggfunc='count')
    .plot.pie(y='id', autopct='%1.0f%%', figsize=(10,10), 
              label='Процентное отношение разрешения на отсылку рекламных сообщений')
)
plt.title('Диаграмма разрешений на сообщения')
plt.show()

In [ ]:
# Процентное отношение и количество
f_countplot(
    f = market_file,
    n = 'разрешить_сообщать',
    suptitle = 'Количество и процент разрешить_сообщать',
    xlabel='Категории',
    ylabel='Количество и процент'
)

**Анализ market_file['популярная_категория']**

In [ ]:
# Диаграмма
(
market_file
    .pivot_table(index='популярная_категория',values='id',aggfunc='count')
    .plot.pie(y='id', autopct='%1.0f%%', figsize=(10,10))              
)
plt.title('Диаграмма популярности категорий')
plt.legend().remove()
plt.show()

In [ ]:
# Процентное отношение и количество
f_countplot(
    f = market_file,
    n = 'популярная_категория',
    suptitle = 'Количество и процент популярная_категория',
    xlabel='Категории',
    ylabel='Количество и процент'
)

Анализ `market_money`. 
В соответствии с заданием, нужно отобрать клиентов с покупательской активностью не менее трёх месяцев, то есть таких, которые что-либо покупали в этот период.

Оставим в таблице только клиентов которые активны все три месяца.

In [ ]:
market_money = market_money[market_money.id.isin(market_money.query('выручка==0')['id'].unique())==False] 
market_money.info()

Анализ числовых значений `market_money`

In [ ]:
# Раздельный вывод
plt.subplots(figsize = (10,5))
sns.histplot(data=market_money, x='выручка', bins=100)
plt.title('Гистограмма суммы выручки за период')
plt.xlabel('Сумма выручки за период')
plt.ylabel('Количество покупателей')
plt.show()

plt.subplots(figsize = (10,5))
sns.boxplot(x=market_money['выручка'])
plt.title('Boxplot суммы выручки за период')
plt.xlabel('Сумма выручки за период')
plt.ylabel('выручка')
plt.show()

In [ ]:
# Совместный график
f_hist_box(data=market_money['выручка'],
           bins=100,
           xlabel='Сумма выручки за период',
           ylabel='Количество покупателей'
)

In [ ]:
market_money.sort_values(by='выручка', ascending=False).head()

Наблюдаем аномально большой выброс. Это мог быть оптовый покупатель либо ошибка ввода данных. Удалим.

In [ ]:
market_money['выручка'].describe()

In [ ]:
market_money = market_money[market_money['выручка'] < 8000]
market_money['выручка'].describe()

In [ ]:
# Раздельный вывод
plt.subplots(figsize = (10,5))
sns.histplot(data=market_money, x='выручка', bins=100)
plt.title('Гистограмма суммы выручки за период')
plt.xlabel('Сумма выручки за период')
plt.ylabel('Количество покупателей')
plt.show()

plt.subplots(figsize = (10,5))
sns.boxplot(x=market_money['выручка'])
plt.title('Boxplot суммы выручки за период')
plt.xlabel('Сумма выручки за период')
plt.ylabel('выручка')
plt.show()

In [ ]:
# Совместный график
f_hist_box(data=market_money['выручка'],
           bins=100,
           xlabel='Сумма выручки за период',
           ylabel='Количество покупателей'
)

Анализ строковых значений `market_money`

In [ ]:
# Диаграмма
(
market_money
    .pivot_table(index='период', values='id',aggfunc='count')
    .plot.pie(y='id', autopct='%1.0f%%', figsize=(10,7), 
              #label='Процентное отношение периодов выручки'
             )
)
plt.title('Выручка по периодам')
plt.legend(bbox_to_anchor=(1, 0.8))
plt.show()

In [ ]:
# Процентное отношение и количество
f_countplot(
    f = market_money,
    n = 'период',
    suptitle = 'Количество и процент по периодам выручки',
    xlabel='Категории',
    ylabel='Количество и процент'
)

Анализ числовых значений `market_time`

In [ ]:
# Рфздельный вывод
plt.subplots(figsize = (10,5))
sns.histplot(data=market_time, x='минут', bins=20)
plt.title('Гистограмма время проведенное покупателем на сайте')
plt.xlabel('Время в минутах')
plt.ylabel('Количество покупателей')
plt.show()

plt.subplots(figsize = (10,5))
sns.boxplot(x=market_time['минут'])
plt.title('Boxplot время проведенное покупателем на сайте')
plt.xlabel('Время в минутах')
plt.ylabel('минут')
plt.show()

In [ ]:
# Совместный график
f_hist_box(data=market_time['минут'],
           bins=20,
           xlabel='Сумма выручки за период',
           ylabel='Количество покупателей'
)

In [ ]:
f_box_hist_relative(data=market_time['минут'],
                    xlabel='Сумма выручки за период',
                    ylabel='Количество покупателей',
                    bins=20,
                    )

Анализ строковых значений `market_time`

In [ ]:
# Диаграмма
(
market_time
    .pivot_table(index='период', values='id',aggfunc='count')
    .plot.pie(y='id', autopct='%1.0f%%', figsize=(10,7), 
              label='Процентное отношение периодов фиксации времени на сайте')
)
plt.title('Период фиксации времени на сайте')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.8))
plt.show()

In [ ]:
# Процентное отношение и количество
f_countplot(
    f = market_time,
    n = 'период',
    suptitle = 'Количество и процент проведенного времени на сайте',
    xlabel='Категории',
    ylabel='Количество и процент'
)

Анализ строковых значений `money`

In [ ]:
# Раздельно
plt.subplots(figsize = (10,5))
sns.histplot(data=money, x='прибыль', bins=70)
plt.title('Гистограмма среднемесячной прибыли от покупателей за 3 месяца')
plt.xlabel('Прибыль')
plt.ylabel('Количество покупателей')
plt.show()

plt.subplots(figsize = (10,5))
sns.boxplot(x=money['прибыль'])
plt.title('Boxplot среднемесячной прибыли от покупателей за 3 месяца')
plt.xlabel('Прибыль')
plt.ylabel('минут')
plt.show()

In [ ]:
# Совместный график
f_hist_box(data=money['прибыль'],
           bins=70,
           xlabel='Сумма выручки за период',
           ylabel='Количество покупателей'
)

In [ ]:
f_box_hist_relative(data=money['прибыль'],
                    xlabel='Сумма выручки за период',
                    ylabel='Количество покупателей',
                    bins=70,
                    )

***Вывод по исследовательскому анализу данных***

Обнаружены выбросы в большинстве данных. Выбросы могут быть связаны с сезонностью продаж, особенностью активности отдельных покупателей, также не исключены ошибки ввода данных. Аномально большой выброс был заполнен медианой.
Из результатов исследования данных видим, что большая доля покупок состоит из товаров для детей (25%), 74% клиентов согласились на рекламную рассылку, примерно столько же имеют стандартный тип сервиса, и 62% сохранили свою покупательную активность. 

Активные пользователи за последние 3 месяца отобраны, полный перечень находится в таблице market_money (1297 покупателей).

Можем провести дальнейшие исследования.

##  Объединение таблиц

###  Подготовка таблиц перед объединением. 

- Подготовка `market_file`

In [ ]:
market_file.info()

- Подготовка `market_time`

In [ ]:
print(market_time.info(), '\n')

df_1 = market_time[market_time['период']=='предыдущий_месяц'].copy()
df_1.rename(columns={'минут':'минут_предыдущего_месяца'},inplace=True)
df_1.drop('период', axis= 1 , inplace= True )

df_2 = market_time[market_time['период']=='текущий_месяц'].copy()
df_2.rename(columns={'минут':'минут_текущего_месяца'},inplace=True)
df_2.drop('период', axis= 1 , inplace= True )

df_market_time = pd.merge(df_1, df_2, on='id', sort=True)

print(df_market_time.info())
df_market_time.head()

- Подготовка `market_time`

In [ ]:

df_1 = market_money[market_money['период']=='текущий_месяц'].copy()
df_1.rename(columns={'выручка':'выручка_предыдущего_месяца'},inplace=True)
df_1.drop('период', axis= 1 , inplace= True )
df_2 = market_money[market_money['период']=='предыдущий_месяц'].copy()
df_2.rename(columns={'выручка':'выручка_текущего_месяца'},inplace=True)
df_2.drop('период', axis= 1 , inplace= True )
df_3 = market_money[market_money['период']=='препредыдущий_месяц'].copy()
df_3.rename(columns={'выручка':'выручка_препредыдущего_месяца'},inplace=True)
df_3.drop('период', axis= 1 , inplace= True )
df_market_money = pd.merge(df_1, df_2, on='id', sort=True)
df_market_money = pd.merge(df_3, df_market_money, on='id', sort=True)
print(df_market_money.info())
df_market_money.head()

### Объединение таблиц

In [ ]:
df = pd.merge(df_market_money, pd.merge(df_market_time, market_file, on='id', sort=True), on='id', sort=True)
print(df.info())
df.head()

***Вывод по объединению таблиц***

Данные успешно объедены, в `df` проведено преобразование данных:
+ столбец `выручка` удален созданы новые столбцы:
    + `выручка_предыдущего_месяца`
    + `выручка_текущего_месяца`
+ столбец `минут` удален созданы новые столбцы:
    + `минут_предыдущего_месяца`
    + `минут_текущего_месяца`


## Корреляционный анализ

In [ ]:
df = df.sort_values(by=['id']).set_index('id')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))
plt.suptitle('Матрица корреляции Спирмена количественных значений')
#sns.heatmap(df.corr(method="spearman", numeric_only=True), annot=True, cmap='coolwarm');
sns.heatmap(df.corr(method="spearman"), annot=True, cmap='coolwarm');

Построим тепловую карту корреляции phi(k) для количественных данных распределение которых ненормальное с выбросами.

In [ ]:
# В interval_cols передадим только непрерывные признаки
interval_cols = ['выручка_препредыдущего_месяца',
                 'выручка_предыдущего_месяца', 
                 'выручка_текущего_месяца',
                 'длительность',
                 'акционные_покупки'
                 ]
fig, ax = plt.subplots(figsize=(11, 10))
plt.suptitle('Матрица корреляции phi(k) всех столбцов')
sns.heatmap(df.phik_matrix(interval_cols=interval_cols), annot=True, cmap='coolwarm');

**Вывод по корреляционному анализу**

Между входными параметрами отсутствует связь выше 0,9 следовательно мультиколлинеарность отсутствует.
+ Максимальное значение корреляция Спирмена (непараметрическая мера оценивающая монотонность связи) corr(S) = 0,88
+ Максимальное значение корреляция phi(k) (учитывает нелинейную зависимость и возвращается к коэффициенту корреляции Пирсона в случае бинарного нормального распределения входных данных) phi(k) = 0,84

## Использование пайплайнов

 - Подготовка данных

In [ ]:
X = df.drop(['покупательская_активность'], axis=1)
y = df['покупательская_активность']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify = y)

display(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

le = LabelEncoder()
le.fit_transform(y_train.unique())
print ('Значениям', le.classes_, 'назначено соответственно', le.transform(le.classes_))
y_train = le.transform(y_train)
y_test = le.transform(y_test)

Кодирование целевого признака проведено корректно

Создадим списки с названиями признаков

In [ ]:
ohe_columns = ['популярная_категория']
ord_columns = ['тип_сервиса', 'разрешить_сообщать']

In [ ]:
df['разрешить_сообщать'].unique()

In [ ]:
num_columns = ['выручка_препредыдущего_месяца',       
               'выручка_предыдущего_месяца',          
               'выручка_текущего_месяца',             
               'минут_предыдущего_месяца',            
               'минут_текущего_месяца',               
               'маркет_актив_тек_мес',
               'маркет_актив_6_мес',
               'длительность',                        
               'акционные_покупки',                   
               'средний_просмотр_категорий_за_визит',
               'неоплаченные_продукты_штук_квартал',  
               'ошибка_сервиса',                      
               'страниц_за_визит']

In [ ]:
# создаём пайплайн для подготовки признаков из списка ohe_columns: заполнение пропусков и OHE-кодирование
# SimpleImputer + OHE
ohe_pipe = Pipeline(
    [('simpleImputer_ohe', SimpleImputer(missing_values=np.nan, strategy='most_frequent')),
     ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]
    )

In [ ]:
# создаём пайплайн для подготовки признаков из списка ord_columns: заполнение пропусков и Ordinal-кодирование
# SimpleImputer + OE
ord_pipe = Pipeline(
    [('simpleImputer_before_ord', SimpleImputer(missing_values=np.nan, strategy='most_frequent')),
     ('ord',  OrdinalEncoder(
                categories=[
                    ['стандарт', 'премиум'],
                    ['да', 'нет'],
                    
                ], 
                handle_unknown='use_encoded_value', unknown_value=np.nan
            )
        ),
     ('simpleImputer_after_ord', SimpleImputer(missing_values=np.nan, strategy='most_frequent'))
    ]
)
# создаём общий пайплайн для подготовки данных
data_preprocessor = ColumnTransformer(
    [('ohe', ohe_pipe, ohe_columns),
     ('ord', ord_pipe, ord_columns),
     ('num', StandardScaler(), num_columns)
    ], 
    remainder='passthrough'
)

### Обучим модели: KNeighborsClassifier(), DecisionTreeClassifier(), LogisticRegression() и SVC().

***Для сравнения результатов работы обученных моделей необходимо выбрать универсальную метрику. Такой является ROC-AUC - метрика модели машинного обучения, которая отображаеть истинную и постоянную способность модели к прогнозированию. У хорошей модели классификации показатель ROC-AUC > 0.9, и в общем случае должен стремиться к 1. При этом указанное значение должно выбираться по значению на тестовой выборке.***

   + Кривая ROC отображает частоту истинных положительных и ложных срабатываний при различных порогах классификации, тогда как AUC показывает совокупную меру производительности модели машинного обучения по всем возможным порогам классификации.
   + ROC означает «Receiver Operating Characteristic» (кривая рабочих характеристик приемника). Это – график, который показывает эффективность модели машинного обучения при решении задачи классификации, отображая частоту истинных срабатываний и частоту ложных срабатываний. AUC расшифровывается как «Area Under the Curve» (площадь под кривой). Она используется для измерения всей площади под кривой ROC.



In [ ]:
# создаём итоговый пайплайн: подготовка данных и модель
pipe_final = Pipeline([
    ('preprocessor', data_preprocessor),
    ('models', KNeighborsClassifier())
])

param_grid = [
    # словарь для модели SVC()
    {
        'models': [KNeighborsClassifier()],
        'models__n_neighbors': range(2, 20),
        'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'],
    },
    {
        'models': [DecisionTreeClassifier(random_state=RANDOM_STATE)],
        'models__max_depth': range(1, 500),
        'models__max_features': range(1, 20),
        'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'],
    },
    {
        'models': [LogisticRegression(random_state=RANDOM_STATE, solver='liblinear',penalty='l1')],
        'models__C': range(1, 15),
        'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'],
    },
    {
        'models': [SVC(random_state=RANDOM_STATE, probability=True)],
        'models__kernel': ['poly', 'rbf', 'sigmoid'],
        'models__degree': range(1, 15),
        'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler(), 'passthrough'],
    }
]

model = RandomizedSearchCV(
                                pipe_final,
                                param_grid, 
                                cv=5,
                                scoring='roc_auc',
                                random_state=RANDOM_STATE,
                                n_jobs=-1
).fit(X_train, y_train)

print('Лучшая модель и её параметры:\n\n', model.best_estimator_)
print ('Метрика ROC-AUC лучшей модели на кросс - валидации:', model.best_score_)

In [ ]:
print('Площадь ROC-кривой на кросс-валидации:', model.best_score_)
roc_auc_test = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print('Площадь ROC-кривой на тестовой выборке:', roc_auc_test)
roc_auc_km = (roc_auc_test - model.best_score_) / model.best_score_*100
print(f'Потеря качества на тестовой выборке  = {-roc_auc_km:.2f} %')

**Вывод по использованию пейплайнов и выбора лучшей модели**

Лучшая модель: SVC(degree=9, kernel='sigmoid', probability=True, random_state=42)

Параметры модели удвлетворительные:
+ Площадь ROC-кривой на тренировочной выборке: 0.840
+ Площадь ROC-кривой на тестовой выборке: 0.843

##  Анализ важности признаков

***Оценим важность признаков для лучшей модели и построим график важности с помощью метода SHAP***

In [ ]:
best_model = model.best_estimator_
best_preprocessor = model.best_estimator_.named_steps['preprocessor']

X_train_trans = pd.DataFrame(best_preprocessor.transform(X_train), columns=best_preprocessor.get_feature_names_out())
#X_train_trans = X_train_trans.sample(frac=0.1, replace=True, random_state=RANDOM_STATE)
X_test_trans = pd.DataFrame(best_preprocessor.transform(X_test), columns=best_preprocessor.get_feature_names_out())
#X_test_trans = X_test_trans.sample(frac=0.1, replace=True, random_state=RANDOM_STATE)

#explainer = shap.KernelExplainer(best_model.named_steps['models'].predict, X_train_trans)
##shap_values = explainer.shap_values(X_test_trans, nsamples=100)
#shap_values = explainer(X_test_trans)

explainer = shap.SamplingExplainer(best_model.named_steps['models'].predict, X_train_trans)
#shap_values = explainer.shap_values(X_test_trans, nsamples=100, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(X_test_trans)
shap_values_1 = explainer(X_test_trans)


In [ ]:
shap.initjs()
shap.summary_plot(shap_values, X_test_trans, show=False)

# Получение текущих объектов
fig, ax = plt.gcf(), plt.gca()

# Изменение основных параметров графика
ax.tick_params(labelsize=14)
ax.set_xlabel("SHAP уровень влияния на выходные признаки", fontsize=14)
ax.set_title('Важность признаков лучшей модели', fontsize=16)

# Получить цветовую панель
cb_ax = fig.axes[1] 

# Изменение параметров цветовой панели
cb_ax.tick_params(labelsize=15)
cb_ax.set_ylabel("Уровень важности признаков", fontsize=20)

plt.show()

In [ ]:
shap.initjs()
shap.summary_plot(shap_values, X_test_trans, plot_type='bar', show=False)

fig, ax = plt.gcf(), plt.gca()
ax.tick_params(labelsize=14)
ax.set_xlabel("SHAP уровень влияния на выходные признаки", fontsize=14)
ax.set_title('Важность признаков лучшей модели', fontsize=16)
cb_ax = fig.axes[0] 
cb_ax.tick_params(labelsize=15)
cb_ax.set_ylabel("Уровень важности признаков", fontsize=20)
plt.show();

In [ ]:
# Визуализация влияния признаков на таргет
shap_values_sum = 0
for i in range(0, len(shap_values_1)):
    shap_values_sum += shap_values_1[i]
    
shap.initjs()
fig, ax = plt.gcf(), plt.gca()
plt.sca(ax)

#shap.plots.waterfall((shap_values_sum.abs / len(shap_values)), max_display=20) 
shap.plots.waterfall((shap_values_sum / len(shap_values_1)), max_display=20) 
#shap.force_plot(explainer.expected_value, shap_values_sum.values/len(shap_values_1), X_test_trans.iloc[0, :], matplotlib=True)

ax.tick_params(labelsize=14)
ax.set_xlabel("SHAP уровень влияния на выходные признаки", fontsize=14)
ax.set_title('Важность признаков лучшей модели', fontsize=16)
cb_ax = fig.axes[0] 
cb_ax.tick_params(labelsize=15)
cb_ax.set_ylabel("Уровень важности признаков", fontsize=20)
plt.show();

**Выводы о значимости признаков:**

Отобрана лучшая модель: `SVC(degree=9, kernel='sigmoid', probability=True, random_state=42)`

Наиболее важные признаки (по убывающей):
+ 'акционные покупки'
+ 'средний_просмотр_категорий_за_визит'
+ 'минут_текущегого_месяца'
+ 'страниц_за_визит'
+ 'маркет_актив_6_мес' 

Менее важные признаки :
+ 'популярная категория_Техника для красоты и здоровья
+ 'популярная_категория_Товары для детей'
+ 'разрешить_сообщать'
+ 'популярная_категория_Косметика и аксесуары'
+ 'маркет_актив_тек_мес'

##  Сегментация покупателей

### Выполним сегментацию покупателей. Используем результаты моделирования и данные о прибыльности покупателей

Сделаем прогноз модели с выбранным порогом и вероятностями

Учитывая отношения процентного распределения данных в нормальном распределенн по значениям стандартного отклонения определим, что значения попадающие в порог 
+ +- 1 СКО от медианного значения являются нормальными их объем примерно составит 68,26 %
+ ниже этого порогога низкий доход, а выше большой или примерно по 15,87 %

В данном случае пользуемся именно таким распределением, потому что не имеем возможности уточнить эти значения у заказчика.

- Создадим дополнительный столбец в `money`
- Сделаем объединение таблиц

In [ ]:
med = money['прибыль'].median()
sko = np.std(money['прибыль'])
print("Медианное значение : ", med)
print("Стандартное отклонение : ", sko)
money['прибыль'].describe()

In [ ]:
# Функция расчета предсказаний с учетом изменяемого порога
# Использование функции predict_proba взамен predict
def custom_predict(X, threshold):
    probs = best_model.predict_proba(X) 
    return (probs[:, 1] > threshold).astype(int)

In [ ]:
# предсказание на данных с выбранным порогом
predictions_precision = custom_predict(X, threshold=0.5)

In [ ]:
X['predictions'] = predictions_precision
X['probs'] = best_model.predict_proba(X)[:,1]

In [ ]:
money['категории_прибыли'] = money['прибыль'].apply(lambda x: 'мало' if x < (med - sko) 
                                                       else ('много' if x > (med + sko) 
                                                       else 'нормально'))

In [ ]:
# Вернем индекс в таблицу для их объединения
X = X.reset_index()

In [ ]:
X = pd.merge(X, money, on='id', sort=True)
X.head()

Построим диаграмму рассеяния вероятности "Снижения покупательской активности" и прибыли для визуализации границы разделения на группы покупательской активности

In [ ]:
plt.figure(figsize=[15, 5])
plt.scatter(X.probs, X.прибыль);
plt.title('Диаграмма рассеяния вероятности "Снижения покупательской активности" и прибыли')
plt.xlabel('Вероятность снижения покупательской активности')
plt.ylabel('Прибыль, %')
plt.show()

Диаграмма не дала достаточного понимания распределения, но скорее носит распределенный характер с провалом в среднем значении подтвердим это построиф график распределения порога предсказания

In [ ]:
plt.figure(figsize=[15, 5])
sns.histplot(data=X, x='probs', binwidth=0.025)
plt.title('Распределение порога предсказания')
plt.xlabel('Порог предсказания')
plt.ylabel('Количество покупателей')
plt.show()
plt.show();

**Наша цель:**\
Клиентам, у которых снизилась покупательская активность, сделать персональное предложение и вернуть их за покупками.

Введем котегориальный столбец в данные, для сравнения данных.\
Определим границу разделения 0.79 так как диаграмма представляет собой "Бабочку" и количество значений резко возрастает от этого порога

In [ ]:
X['категория_снижения'] = X['probs'].apply(lambda x: 'Снизилась' if x > 0.79 else 'Прежний уровень')                          

### Проведем исследование группы покупателей. Сделаем предложения по работе с сегментом для увеличения покупательской активности

Для работы выделим группу с малым доходом и высокой вероятностью снижения покупательской способности, для сравнения оставим группу с прежним уровнем активности. Она покажет целевой уровень, куда нужно стремиться и какие решения принимать в разрезе важных признаков, определенных на 7 шаге работы.

In [ ]:
x=['акционные_покупки',
   'страниц_за_визит',
   'минут_предыдущего_месяца',
   'минут_текущего_месяца',
   'неоплаченные_продукты_штук_квартал',
   'средний_просмотр_категорий_за_визит'
  ]

y=['акционные_покупки'
  ]

g = sns.pairplot(X.query('категории_прибыли == "мало"'), hue='категория_снижения', x_vars=x, y_vars=y)
g.fig.suptitle("Диаграммы рассеяния покупателей с большой долей акционных покупок в разрезе прибыльности", y=1.2, fontsize=15)
plt.show()

**Выделим группу клиентов, количество покупок акционных товаров не превышают 3 штук.**\
Сформулируем предложение по работе с сегментом для увеличения покупательской активности.

In [ ]:
x=['акционные_покупки',
   'страниц_за_визит',
   'минут_предыдущего_месяца',
   'минут_текущего_месяца',
   'неоплаченные_продукты_штук_квартал',
   'средний_просмотр_категорий_за_визит'
  ]
y=['акционные_покупки'
  ]

g = sns.pairplot(X.query('категории_прибыли == "мало" & акционные_покупки < 3'), hue='категория_снижения', x_vars=x, y_vars=y)
g.fig.suptitle("Диаграммы рассеяния покупателей с большой долей акционных покупок в разрезе прибыльности", y=1.2, fontsize=15)
plt.show()

 ### Определенные сегменты и их анализ

 **Группа клиентов, количество акционных товаров у которых не превышает 3 штук**

В этом сегменте наблюдается снижение покупательской активности. Активность клиентов особенно снижается у тех, у кого количество акционных покупок меньше 3, а у тех кто сделал 3 покупки уже могут попасть в среду активных пользователей. Возможно, это связано с уменьшением рекламных кампаний по акциям.

**Рекомендации:**

- Выделить этих клиентов в отдельную рабочую группу и для увеличения их активности:
  - Активнее предлагать акционные товары и проводить уведомления, рекламные кампании.
  - Удерживать их внимание на страницах сайта: замечено, что при просмотре более 6 страниц покупательская активность остается на прежнем уровне.
  - Увеличение числа просмотренных страниц приведет к увеличению времени, проведенного за визит. Если визит длится более 10 минут, вероятность сохранения покупательской активности повышается.

### Группа клиентов с высокой прибыльностью и более 4 неоплаченных покупок в корзине**

В этом сегменте также наблюдается значительное снижение покупательской активности.

**Рекомендации:**

- Выделить этих клиентов в отдельную рабочую группу и для увеличения их активности:
  - Уведомлять их, если в корзине более 6 неоплаченных товаров. Это может вернуть их на сайт, повысить важные метрики и увеличить покупательскую активность.


**Выделим группу клиентов, с высокой  прибыльностью и количества неоплаченных покупок в корзине больше 4**\
Сформулируем предложение по работе с сегментом для увеличения покупательской активности.

Прибыль от клиентов распределена нормально. Определим, что наиболее высокая прибыль клиентов лежит за 3 квантилем нормального распределения.

In [ ]:
x=['акционные_покупки',
   'страниц_за_визит',
   'минут_предыдущего_месяца',
   'минут_текущего_месяца',
   'неоплаченные_продукты_штук_квартал',
   'средний_просмотр_категорий_за_визит'
  ]
y=['неоплаченные_продукты_штук_квартал'
  
  ]

g = sns.pairplot(X.query('категории_прибыли == "много" & неоплаченные_продукты_штук_квартал > 4'), hue='категория_снижения', x_vars=x, y_vars=y)
g.fig.suptitle("Диаграммы рассеяния покупателей с количеством неоплаченных товаров в корзине больше 4", y=1.2, fontsize=15)
plt.show()

**Промежуточный итог**

В сегменте присутствует значительная доля снижения покупательской активности клиентов.

Предлагается выделить таких клиентов в отдельную рабочую группу и для учеличения их активности:
+ при количестве неоплаченных продуктов в корзине болше 6, уведомлять об этом покупателя, что возможно приведет покупателя обратно на сайт, повысит остальные важные метрики и увеличит их покупательную активность.

В общем случае рекомендации такие же как и для предыдущей группы.

### Подведем итоги о сегментах

 ***Определенные сегменты и их анализ***

**Группа клиентов, количество акционных товаров у которых не превышает 3 штуки**

В этом сегменте наблюдается снижение покупательской активности, особенно у клиентов, у которых количество акционных покупок не превышает трех. Возможно, это связано с уменьшением рекламных кампаний по акциям.

**Рекомендации:**

- Выделить этих клиентов в отдельную рабочую группу.
- Активнее предлагать им акционные товары и проводить рекламные кампании.
- Удерживать их внимание на страницах сайта: замечено, что при просмотре более 6 страниц покупательская активность сохраняется.
- Увеличение числа просмотренных страниц приведет к увеличению времени, проведенного на сайте. При проведении на сайте более 10 минут вероятность сохранения покупательской активности повышается.

**Группа клиентов с высокой прибыльностью и более 4 неоплаченных покупок в корзине**

В этом сегменте также наблюдается значительное снижение покупательской активности.

**Рекомендации:**

- Выделить этих клиентов в отдельную рабочую группу.
- Уведомлять их, если в корзине более 6 неоплаченных товаров. Это может вернуть их на сайт, повысить важные метрики и увеличить покупательскую активность.

Общие рекомендации для обеих групп заключаются в том, чтобы активно предлагать акционные товары, проводить рекламные кампании, удерживать внимание на страницах сайта и уведомлять о неоплаченных товарах в корзине.

## Общий вывод

**Задача исследования**

Целью исследования было разработать решение для персонализации предложений постоянным клиентам, чтобы увеличить их покупательскую активность. Для анализа были предоставлены следующие данные:

- **market_file.csv** — данные о поведении покупателя на сайте, коммуникациях с ним и его продуктовом поведении.
- **market_money.csv** — данные о выручке, получаемой магазином от покупателя за период взаимодействия.
- **market_time.csv** — данные о времени, проведенном покупателем на сайте в течение периода.
- **money.csv** — данные о среднемесячной прибыли покупателя за последние три месяца.

**Предобработка данных**

1. Изменены названия столбцов: перевод в нижний регистр и замена пробелов на подчеркивания.
2. В **market_file**:
   - В колонке "тип_сервиса" данные "стандартт" заменены на "стандарт".
   - Изменены типы столбцов: "маркет_актив_6_мес" и "акционные_покупки" на float.
3. В **market_money**:
   - Изменен тип столбца "выручка" на float.
4. В **market_time**:
   - В колонке "период" данные "предыдцщий_месяц" заменены на "предыдущий_месяц".

**Анализ данных и подготовка**

- Проведен исследовательский анализ данных.
- Проведена подготовка и объединение таблиц.
- Для поиска лучшей модели был проведен корреляционный анализ данных для исключения мультиколлинеарности.
- Подготовлены пайплайны для отбора лучшей модели в ручном и автоматическом режиме.

**Лучшая модель**

Отобрана лучшая модель: `SVC(degree=9, kernel='sigmoid', probability=True, random_state=42)`

Наиболее важные признаки (по убывающей):
+ 'акционные покупки'
+ 'средний_просмотр_категорий_за_визит'
+ 'минут_текущегого_месяца'
+ 'страниц_за_визит'
+ 'маркет_актив_6_мес' 

Менее важные признаки :
+ 'популярная категория_Техника для красоты и здоровья
+ 'популярная_категория_Товары для детей'
+ 'разрешить_сообщать'
+ 'популярная_категория_Косметика и аксесуары'
+ 'маркет_актив_тек_мес'

**Сегменты клиентов и рекомендации**

**Группа клиентов, у которых количество акционных товаров не превышает 3 штук**

В этом сегменте наблюдается снижение покупательской активности, особенно у клиентов с количеством акционных покупок, менее 3 штук. Возможно, это связано с уменьшением рекламных кампаний по акциям.

***Рекомендации:***

- Выделить таких клиентов в отдельную рабочую группу.
- Активно предлагать им акционные товары и проводить рекламные кампании.
- Задерживать их внимание на страницах сайта: при просмотре более 6 страниц активность остается на прежнем уровне.
- Увеличение числа просмотренных страниц приведет к увеличению времени, проведенного на сайте: при более чем 10 минутах вероятность сохранения покупательской активности повышается.

**Группа клиентов с высокой прибыльностью и более 4 неоплаченных покупок в корзине**

В этом сегменте также наблюдается значительное снижение покупательской активности.
**Рекомендации:**

- Выделить этих клиентов в отдельную рабочую группу и для увеличения их активности:
- Уведомлять их, если в корзине более 6 неоплаченных товаров. Это может вернуть их на сайт, повысить важные метрики и увеличить покупательскую активность.